In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02. DFMGAN Training\n",
    "Training the Discriminator-Filtered Monte Carlo GAN (WGAN-GP).\n",
    "\n",
    "The checkpoint written here is loaded by `MonteCarloPipeline.load_models(checkpoint_path=...)`. Without a trained checkpoint the pipeline uses GBM."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.insert(0, os.path.abspath('..'))\n",
    "\n",
    "import torch\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from src.pipeline.data_fetcher import DataFetcher\n",
    "from src.pipeline.dfmgan.generator import DFMGANConfig, ConditionalMCGenerator, TimeGANDiscriminator\n",
    "from src.pipeline.dfmgan.trainer import DFMGANTrainer\n",
    "\n",
    "%matplotlib inline\n",
    "torch.manual_seed(42)\n",
    "np.random.seed(42)\n",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "print('[INFO] Using device:', device)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fetcher = DataFetcher()\n",
    "tickers = ['AAPL', 'MSFT', 'GOOGL']\n",
    "\n",
    "data = fetcher.get_historical_data(tuple(tickers), period='5y')\n",
    "if data is None or data.shape[1] != len(tickers):\n",
    "    raise RuntimeError('Need data for ALL tickers to train. Provider errors: ' + str(fetcher.last_errors))\n",
    "\n",
    "data = data[tickers].dropna()\n",
    "print('[INFO] Data loaded:', data.shape)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "seq_len = 60\n",
    "values = data.values\n",
    "if len(values) <= seq_len:\n",
    "    raise RuntimeError('Not enough rows to build sequences')\n",
    "\n",
    "sequences = np.stack([values[i:i + seq_len] for i in range(len(values) - seq_len)])\n",
    "real_data = torch.tensor(sequences, dtype=torch.float32).to(device)\n",
    "print('[INFO] Real data shape:', tuple(real_data.shape))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# latent_dim / hidden_dim MUST equal the DFMGANConfig defaults: MonteCarloPipeline.load_models()\n",
    "# rebuilds the network from those defaults before loading the checkpoint.\n",
    "config = DFMGANConfig(n_assets=len(tickers), seq_len=seq_len, latent_dim=32, hidden_dim=64, n_simulations=1000)\n",
    "\n",
    "generator = ConditionalMCGenerator(config).to(device)\n",
    "discriminator = TimeGANDiscriminator(config).to(device)\n",
    "\n",
    "print('Generator parameters:', sum(p.numel() for p in generator.parameters()))\n",
    "print('Critic parameters:', sum(p.numel() for p in discriminator.parameters()))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "trainer = DFMGANTrainer(generator, discriminator, config, device=device)\n",
    "history = trainer.train(real_data=real_data, epochs=50, batch_size=32)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(10, 4))\n",
    "plt.plot(history['d_loss'], label='Critic loss (with gradient penalty)')\n",
    "plt.plot(history['g_loss'], label='Generator loss')\n",
    "plt.xlabel('Epoch')\n",
    "plt.title('WGAN-GP training losses (no accuracy metric exists for a Wasserstein critic)')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "generator.eval()\n",
    "with torch.no_grad():\n",
    "    generated = generator(real_data[:1, :30, :], n_paths=100)\n",
    "print('[INFO] Generated paths shape:', tuple(generated.shape))\n",
    "\n",
    "# Sanity check: are generated daily volatilities in the same ballpark as real ones?\n",
    "real_ret = np.diff(np.log(real_data.cpu().numpy()), axis=1)\n",
    "gen_ret = np.diff(np.log(generated.cpu().numpy()), axis=1)\n",
    "print('real daily vol     :', real_ret.std(axis=(0, 1)).round(4))\n",
    "print('generated daily vol:', gen_ret.std(axis=(0, 1)).round(4))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "real_plot = real_data[0, :, 0].cpu().numpy()\n",
    "gen_plot = generated[:10, :, 0].cpu().numpy()\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "for i in range(10):\n",
    "    plt.plot(gen_plot[i], alpha=0.3)\n",
    "plt.plot(real_plot, 'k--', label='Real', linewidth=2)\n",
    "plt.xlabel('Time steps')\n",
    "plt.ylabel('Price')\n",
    "plt.title('Generated vs real (' + tickers[0] + ')')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "os.makedirs('../data/models', exist_ok=True)\n",
    "trainer.save('../data/models/dfmgan.pth')\n",
    "print('[INFO] Saved ../data/models/dfmgan.pth')\n",
    "\n",
    "# Verify the pipeline can actually load it (use_gan flips back to False if loading fails)\n",
    "from src.pipeline import MonteCarloPipeline\n",
    "pipe = MonteCarloPipeline(n_assets=len(tickers), seq_len=seq_len, use_gan=True, use_live_data=False)\n",
    "pipe.load_models(checkpoint_path='../data/models/dfmgan.pth')\n",
    "print('GAN active in pipeline:', pipe.use_gan)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}